# PulmoScan: Kaggle training

Train **DenseNet121 + EfficientNet-B0** with the same audited split/preprocessing/training code as the local project.
This notebook embeds the necessary source files. **No project ZIP, private repository, API token, `.env`, or local dataset upload is required.**

Before running:
1. In Kaggle, import this `.ipynb` using **Import Notebook**.
2. **Add Input**: [paultimothymooney/chest-xray-pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia).
3. In session settings choose an available **GPU** and enable **Internet** for official ImageNet weights.
4. Run cells in order. GPU availability/account requirements depend on Kaggle. Only one GPU is used.

The input dataset stays read-only. Models/logs are written to `/kaggle/working/pulmoscan_training/`.
Download `pulmoscan_models.zip` when complete, and **Save Version** with outputs before ending the session.
No app server starts, no live model is replaced, and test evaluation is disabled by default.
[Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks).

In [ ]:
import base64, hashlib, json, os, sys, zlib
from pathlib import Path, PurePosixPath

WORKSPACE = Path('/kaggle/working/pulmoscan_training')
SOURCE_INDEX = {'src/compute.py': '14e5d99ba467b12c2352d19c0272498d53a0ad537119211208422e3b9bd8ea6a', 'src/data_protocol.py': '2df0eb1dd2d353cf2a0cb61edd8210e4740bd6627a3fb51ed7a4c0693ad3b88a', 'src/modeling.py': 'f19897e3415d385c569275b0da3b663c8caf70fa16d7b9e29348010463003ef9', 'src/scoring.py': '8b221e04257f9a3200b56c2e92ce11c574e8c3818193fe54911d3a1e74a27d19', 'src/training.py': '9c54f8fa4ed1bd7ad22c30b342cbc981735760119b7da891404265762af73032', 'src/prepare_dataset.py': '0bbe8eee5864339f5517fe4d0e9a4572fbbb800504ae7797ee6124fb0953d1b9', 'scripts/train_models.py': '77ed9b6829a31f4fe676f39ffff705dfbd3d06af14da89a80093366fab509040', 'scripts/audit_dataset.py': '607c7d520f802558125d3485a8f8e7c1ac25f64b518fda9198e641970324ba54', 'scripts/evaluate_experiment.py': 'de00f319546b7517ea06e2c28ec6ca840afbf521f691655241cb6ba4b8e1dc55', 'scripts/check_training_data.py': '2fb9915536f07918ab0291f7923175a6afb01f7c4c0d60bc516227b125009b87', 'scripts/experiment_bundle.py': '2b5f0b86a54483df6ed53c957f8a963ee2859d8053ffee7700e18001a2311106', 'scripts/diagnose_experiment.py': '3f8120d243c4cc0e81cefabc58f3af4f8bc3b5a96b455d00e11e17de2039286f', 'kaggle/runtime.py': 'f1afcc5f19754f1745041637dc937fd7c7eabb2c73eb3ac9af4a9a3228f2737f', 'kaggle/train_config.json': 'a7e7ddc67576407d0975e399f30a0e9ab8b44d538fceb4534a834a581d5f9af5', 'kaggle/train_config_v2.json': '556c5040986be7bd77191445bdf86fedba09060e13349f33f6be86fad3f95ad0', 'kaggle/requirements.txt': '132e6f60571116dab005d2a5339e43b2514a6fdfd2a590f17a12e94672c2bf26', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'scripts/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}
SOURCE_PAYLOAD = '...'
packed = base64.b64decode(SOURCE_PAYLOAD)
if hashlib.sha256(packed).hexdigest() != '91504377a14000d2db93f1f4873887ac432a4780b9877fc3b83636a82e8fcf11':
    raise ValueError('Embedded source checksum failed. Re-import an intact notebook.')
files = json.loads(zlib.decompress(packed))
if set(files) != set(SOURCE_INDEX):
    raise ValueError('Embedded source index differs.')
for name, encoded in files.items():
    relative = PurePosixPath(name)
    if relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:
        raise ValueError('Unsafe embedded source path.')
    content = base64.b64decode(encoded)
    if hashlib.sha256(content).hexdigest() != SOURCE_INDEX[name]:
        raise ValueError('Embedded source checksum failed: ' + name)
    target = WORKSPACE.joinpath(*relative.parts)
    if not target.resolve().is_relative_to(WORKSPACE.resolve()):
        raise ValueError('Embedded source escaped workspace.')
    if target.exists() and target.read_bytes() != content:
        raise ValueError('Existing workspace uses different code. Preserve it and choose a new WORKSPACE.')
WORKSPACE.mkdir(parents=True, exist_ok=True)
for name, encoded in files.items():
    target = WORKSPACE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_bytes(base64.b64decode(encoded))
index_path = WORKSPACE / 'source_index.json'
index_text = json.dumps(SOURCE_INDEX, sort_keys=True, indent=2)
if index_path.exists() and index_path.read_text() != index_text:
    raise ValueError('Existing source index differs; keep the original notebook for a resumed run.')
index_path.write_text(index_text)
sys.path.insert(0, str(WORKSPACE))
os.chdir(WORKSPACE)
print('Verified training sources:', len(files), '| workspace:', WORKSPACE)

## Settings

Defaults use physical batch **16**, accumulation **1** (effective batch 16), 224px, up to 25 epochs per model.
If GPU memory is insufficient, use batch 8/accumulation 2 **in a new workspace/run**, preserving the effective batch.
Keep `num_workers=0` for reproducible epoch-boundary resume. Do not change settings when resuming.
`RUN_NAME` stays unchanged when rerunning this cell in the current session.

For a later session: attach your saved notebook output, set `RESUME_SOURCE` to its `pulmoscan_training` directory,
set `RUN_NAME` to the original suite folder, and use the original notebook/code/settings.
Prior Kaggle input mount paths must still exist. Resume uses the last completed epoch, not the middle of an epoch.

In [ ]:
from datetime import datetime, timezone
DATA_ROOT = None  # Auto-detect, or paste the directory directly containing train/, val/, test/.
PROFILE = 'v1'
REQUIRE_GPU = True  # Set False deliberately to allow CPU; local training remains GPU-or-CPU automatic.
BATCH_SIZE = 16
ACCUMULATION_STEPS = 1
EPOCHS = 25
RUN_NAME = globals().get('RUN_NAME', 'kaggle-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RESUME_SOURCE = None  # Example: '/kaggle/input/my-saved-training/pulmoscan_training'
RUN_FINAL_TEST = False  # Leave False until both validation summaries have been reviewed.
INCLUDE_RESUME_IN_ZIP = False  # True adds larger last.pt optimizer/RNG checkpoints to the download.
print('Suite:', RUN_NAME)

## Runtime check

Kaggle supplies torch and its matching GPU torchvision build. This cell keeps that pair installed,
imports the training libraries, and installs only missing small extras. It never installs the local API dependencies.
If torch/torchvision fail to import together, use a compatible Kaggle GPU runtime; do not install Windows CUDA wheels.
Internet must be enabled for first-time official weight downloads and any missing extras.

In [ ]:
import importlib.util, subprocess
extras = {'numpy': 'numpy>=1.24,<3', 'sklearn': 'scikit-learn>=1.3,<2',
          'PIL': 'pillow>=10.4,<14', 'dotenv': 'python-dotenv>=1,<2'}
missing = [requirement for module, requirement in extras.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
# Load by file location to avoid a name collision with the installed Kaggle SDK.
spec = importlib.util.spec_from_file_location('pulmoscan_notebook_runtime', WORKSPACE / 'kaggle/runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
os.environ['PULMOSCAN_DEVICE'] = 'cuda' if REQUIRE_GPU else 'auto'
device = runtime.check_environment(REQUIRE_GPU)
if RESUME_SOURCE and not (WORKSPACE / 'data').exists():
    runtime.restore_previous(RESUME_SOURCE, WORKSPACE)
dataset_root = runtime.discover_dataset('/kaggle/input', DATA_ROOT)
print('Read-only source dataset:', dataset_root)

## Audit and prepare

Fully decode images; record byte/pixel hashes; reserve test-connected filename/duplicate groups;
deduplicate exact pixels; split the remaining pool by candidate groups, stratified by class.
Training-only augmentation is mild; validation/test use deterministic full-field letterboxing.
No source images are copied, modified, or silently skipped. Every read during training checks the saved byte hash.

This is a fresh audit of Kaggle's input, **not the damaged local source tree**. Local-only suspect exclusions
are not carried across, so counts and manifest identity can differ from the laptop experiment.
Filename IDs are heuristic, near-duplicates are not fully verified, and this pediatric benchmark does not establish
clinical validity or external generalization. Never tune on the reserved test set.

In [ ]:
config_path, manifest_path = runtime.prepare(
    WORKSPACE, dataset_root, BATCH_SIZE, ACCUMULATION_STEPS, EPOCHS, profile=PROFILE)
from scripts.check_training_data import main as check_data
if check_data(['--manifest', str(manifest_path), '--passes', '2']):
    raise RuntimeError('Dataset integrity checks failed. Inspect the report; do not bypass hashes.')

## Preflight (no training)

Check source hashes, official pretrained weight hashes, and finite forward outputs on the selected GPU.
This does not establish training peak memory or accuracy.

In [ ]:
from scripts.train_models import cli as training_cli
if training_cli(['--config', str(config_path), '--check']):
    raise RuntimeError('Preflight failed. Inspect the output before training.')

## Train both candidates

This cell performs real training with live batch/epoch logs. DenseNet121 runs first, then EfficientNet-B0.
New profiles select checkpoints using validation ROC-AUC from raw logit margins
(validation loss breaks exact ties), with early stopping. Legacy saved profiles retain probability AUC.
Class weights, warm-up, frozen BatchNorm statistics, mixed precision, gradient clipping, and accumulation are shared
with the local workflow. Initial AMP scale reductions may occur; finite-loss checks remain enabled.
A finalized threshold uses validation only, targeting at least 95% validation sensitivity while maximizing balanced accuracy.
That target is an experiment policy, not guaranteed test sensitivity or clinical validation.

Rerunning this cell skips completed candidates and resumes an incomplete candidate's `last.pt` when available.
Stop/interruption before the first full epoch needs a new `RUN_NAME`. Save outputs before ending a Kaggle session.

In [ ]:
suite = runtime.run_or_resume(WORKSPACE, config_path, RUN_NAME)
print('Training/validation complete:', suite)

## Review validation results

Compare validation results before enabling final test evaluation. Larger GPU resources do not guarantee higher accuracy.
Graphs use the recorded history only; no additional model inference is performed here.

In [ ]:
comparison = json.loads((suite / 'comparison.json').read_text())
for row in comparison['candidates']:
    result = row['validation_at_selected_threshold']
    print(f"{row['architecture']}: best epoch={row['best_epoch']} | AUC={row['selection_metric_value']:.5f} | "
          f"accuracy={result['accuracy']:.3%} | sensitivity={result['sensitivity']:.3%} | "
          f"specificity={result['specificity']:.3%} | threshold={result['threshold']:.6f}")
    if 'calibration_at_selected_threshold' in row:
        print('Separate threshold-calibration holdout:', json.dumps(row['calibration_at_selected_threshold']))
print('Validation-selected checkpoint:', comparison['recommended_checkpoint'])
print('Pending candidates:', comparison['pending_candidates'])
if importlib.util.find_spec('matplotlib'):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for row in comparison['candidates']:
        history = json.loads((suite / row['architecture'] / 'history.json').read_text())
        epochs = [item['epoch'] for item in history]
        auc_field = 'roc_auc_logit_margin' if row.get('selection_metric') == 'val_logit_margin_auc' else 'roc_auc'
        axes[0].plot(epochs, [item['validation'][auc_field] for item in history], label=row['architecture'])
        axes[1].plot(epochs, [item['validation']['loss'] for item in history], label=row['architecture'])
    for axis, title in zip(axes, ('Validation ROC-AUC', 'Validation loss')):
        axis.set(xlabel='Epoch', title=title)
        axis.legend()
        axis.grid(alpha=.2)
    plt.tight_layout()
    plt.show()

## Optional final test — disabled by default

After reviewing the completed validation comparison, set `RUN_FINAL_TEST=True` in the settings cell
and run **only the settings cell and this cell**. Do not rerun training to optimize a test result.
The validation-selected model and threshold are fixed. Existing final results are displayed, not recomputed.
If you skip this step, the ZIP contains validation results only and says test evaluation is pending.

In [ ]:
final_path = suite / 'final_test_results.json'
if RUN_FINAL_TEST and not final_path.exists():
    command = [sys.executable, '-u', 'scripts/evaluate_experiment.py', '--suite', str(suite)]
    if PROFILE == 'v2':
        command.append('--allow-reviewed-benchmark')
    subprocess.run(command, cwd=WORKSPACE, check=True)
if final_path.exists():
    print(json.dumps(json.loads(final_path.read_text())['metrics'], indent=2))
else:
    print('Test not evaluated. Review validation first, then explicitly enable this step.')

## Download models and evidence

The ZIP contains both `best.pt` models, their preprocessing/threshold metadata, split manifests,
histories, logs, validation comparison/predictions, optional final test results, exact source files,
and SHA-256 checksums. `last.pt` is optional; it is always present in the saved workspace after a completed epoch.
Save a notebook version including outputs before stopping the session; a download link alone does not persist a session.

Download `pulmoscan_models.zip` from the link or Kaggle's output file browser.
On your laptop: `python scripts/experiment_bundle.py import --archive "PATH_TO/pulmoscan_models.zip"`.
This verifies and stores the experiment, without replacing the app's current model or starting a server.
EfficientNet serving integration still needs review if EfficientNet wins; the current API supports DenseNet121.

In [ ]:
from scripts.experiment_bundle import export_suite, verify_bundle
download = Path('/kaggle/working/pulmoscan_models.zip')
if download.exists():
    # Explicitly choose a new filename if you exported before adding final test results.
    download = download.with_name('pulmoscan_models_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
export_suite(suite, download, include_resume=INCLUDE_RESUME_IN_ZIP)
receipt = verify_bundle(download)
print('Verified download:', download, '| final test included:', receipt['test_evaluated'])
from IPython.display import FileLink, display
display(FileLink(str(download)))
print('Also available in the Kaggle output browser. Save Version with outputs before ending this session.')